# 3. Missing values and outliers

## Small idea: missingness has meaning

A blank value can mean “unknown,” “not measured,” “not applicable,” or “data-entry
failure.” The treatment should follow that meaning. Statistical imputation is a learned
transformation and must be fitted on training data only.

**Learning goals**

- distinguish unknown from structural missingness;
- impute numerical and categorical columns safely;
- add missingness indicators when useful;
- flag outliers and use robust transformations without target-guided deletion.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, RobustScaler

train = pd.DataFrame({
    "age": [19.0, 25.0, np.nan, 33.0, 29.0, 22.0],
    "tokens": [80.0, 125.0, 110.0, np.nan, 2_400.0, 95.0],
    "task_type": ["free", "picture", None, "free", "social", "picture"],
    "has_dictionary": [True, False, False, True, False, True],
    "dictionary_score": [8.0, np.nan, np.nan, 6.0, np.nan, 7.0],
})

test = pd.DataFrame({
    "age": [np.nan, 41.0],
    "tokens": [140.0, 3_100.0],
    "task_type": ["interview", None],
    "has_dictionary": [False, True],
    "dictionary_score": [np.nan, 9.0],
})

train

## 1. Unknown versus not applicable

A missing `age` is unknown and may be imputed. A missing `dictionary_score` is
structurally expected when `has_dictionary=False`; replacing it with the population
median would erase that meaning. Keep an explicit availability indicator and choose a
documented structural value.

In [ ]:
def encode_structural_missingness(frame: pd.DataFrame) -> pd.DataFrame:
    result = frame.copy()
    result["dictionary_score"] = result["dictionary_score"].where(
        result["has_dictionary"],
        other=0.0,
    )
    return result

train_structured = encode_structural_missingness(train)
test_structured = encode_structural_missingness(test)
train_structured[["has_dictionary", "dictionary_score"]]

## 2. Numerical imputation with an indicator

In [ ]:
numeric_imputer = SimpleImputer(strategy="median", add_indicator=True)
numeric_imputer.fit(train_structured[["age", "tokens"]])

train_numeric = numeric_imputer.transform(train_structured[["age", "tokens"]])
test_numeric = numeric_imputer.transform(test_structured[["age", "tokens"]])

print("Training medians:", numeric_imputer.statistics_)
print("Output names:", numeric_imputer.get_feature_names_out().tolist())
print("Transformed test rows:\n", test_numeric)

The indicator records that the original value was missing. It can be informative when
missingness is systematic, but it does not repair a biased data-collection process.

## 3. Categorical imputation and unknown categories

In [ ]:
categorical_pipeline = Pipeline([
    ("impute", SimpleImputer(strategy="constant", fill_value="Missing")),
    ("encode", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

categorical_pipeline.fit(train_structured[["task_type"]])
transformed_test_categories = categorical_pipeline.transform(test_structured[["task_type"]])

print(categorical_pipeline.get_feature_names_out().tolist())
print(transformed_test_categories)

`interview` appears only in the test data. `handle_unknown="ignore"` preserves the
training feature shape instead of failing or inventing a test-specific column.

## 4. Flag outliers before deciding what to do

In [ ]:
def iqr_flags(series: pd.Series, factor: float = 1.5) -> pd.Series:
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    return ~series.between(q1 - factor * iqr, q3 + factor * iqr)

token_flags = iqr_flags(train_structured["tokens"].dropna())
train_structured.loc[token_flags.index[token_flags], ["tokens", "task_type"]]

Long texts may be genuine and important. Possible responses include correcting a proven
data-entry error, using a log transformation, applying a robust scaler, winsorizing with
documented thresholds, modeling subgroups, or leaving the value unchanged. Do not delete
a row merely because it hurts a target correlation or model score.

In [ ]:
token_train = train_structured[["tokens"]].copy()
token_test = test_structured[["tokens"]].copy()

token_median = token_train["tokens"].median()
token_train_filled = token_train.fillna(token_median)
token_test_filled = token_test.fillna(token_median)

robust_scaler = RobustScaler()
robust_scaler.fit(token_train_filled)

print("Robust-scaled training tokens:\n", robust_scaler.transform(token_train_filled).round(2))
print("Robust-scaled test tokens:\n", robust_scaler.transform(token_test_filled).round(2))

## 5. Put column-specific rules together

In [ ]:
numeric_pipeline = Pipeline([
    ("impute", SimpleImputer(strategy="median", add_indicator=True)),
    ("scale", RobustScaler()),
])

categorical_pipeline = Pipeline([
    ("impute", SimpleImputer(strategy="constant", fill_value="Missing")),
    ("encode", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, ["age", "tokens", "dictionary_score"]),
    ("categorical", categorical_pipeline, ["task_type", "has_dictionary"]),
])

X_train_ready = preprocessor.fit_transform(train_structured)
X_test_ready = preprocessor.transform(test_structured)

print("Training shape:", X_train_ready.shape)
print("Test shape:", X_test_ready.shape)
assert X_train_ready.shape[1] == X_test_ready.shape[1]

## Tiny checkpoint

For each missing value, choose a treatment and justify it: unknown learner age, no
applicable second-language score, unrecorded annotator ID, missing text, and an unseen
task category at prediction time.